## Milestone II - Preprocessing

Complex feature engineering and lagged feature generation.

**Input:** `data/milestone_ii_dataset.pkl` (raw indicators + basic FE from Dataset notebook)

**Output:** `data/milestone_ii_preprocessed.pkl` (fully engineered features ready for modeling)

### Import Packages and Load Data

In [1]:
import pandas as pd
import numpy as np
import warnings

from config import SERIES_CONFIG

warnings.filterwarnings("ignore")

# Load the dataset from the Dataset notebook
df = pd.read_pickle('data/milestone_ii_dataset.pkl')
print(f"Dataset shape: {df.shape}")
print(f"Date range: {df.index.get_level_values('date').min()} to {df.index.get_level_values('date').max()}")
print(f"Countries: {df.index.get_level_values('country').unique().tolist()}")

Dataset shape: (5508, 32)
Date range: 1970-01-01 00:00:00 to 2020-12-01 00:00:00
Countries: ['Australia', 'Canada', 'France', 'Germany', 'Italy', 'Japan', 'South Korea', 'UK', 'USA']


### Feature Engineering

All feature engineering is performed here on the raw indicator dataset.

##### Log Differences (40 features)
For level-based indicators: `real_gdp`, `cpi`, `ind_out`, `oil`, `copper`, `gps`, `vix`, `epu`, `retail_vol`, `national_share_price`
- `{col}_log_1mo`, `{col}_log_3mo`, `{col}_log_6mo`, `{col}_log_12mo`

##### First Differences (12 features)
For rate-based indicators: `unemployment_rate`, `10_yr_yld`, `3_mo_yld`
- `{col}_diff_1mo`, `{col}_diff_3mo`, `{col}_diff_6mo`, `{col}_diff_12mo`

##### Amplitude Deviation Features (10 features)
For amplitude-adjusted indices centered at 100: `comp_consumer_conf`, `cli`
- `{col}_dev`: Deviation from 100 baseline (100 = long-term trend)
- `{col}_diff_1mo`, `{col}_diff_3mo`, `{col}_diff_6mo`, `{col}_diff_12mo`

##### GDP & Technical Recession
- `gdp_qoq_growth`: Quarter-over-quarter GDP growth rate (%)
- `technical_rec`: Boolean flag for two consecutive quarters of negative GDP growth

##### SAHM Rule (3 features)
- `unemployment_3mo_avg`: 3-month rolling average of unemployment rate
- `unemployment_12mo_min`: 12-month rolling minimum of the 3-month average
- `sahm_value`: Difference between the two (signals recession when >= 0.5pp)

##### Rolling Statistics (18 features)
For volatility/market indicators: `vix`, `epu`, `national_share_price`
- `{col}_rolling_mean_3mo`, `{col}_rolling_mean_6mo`, `{col}_rolling_mean_12mo`
- `{col}_rolling_std_3mo`, `{col}_rolling_std_6mo`, `{col}_rolling_std_12mo`

##### Derived Features (4 features)
- `unemployment_accel`: Second derivative of unemployment rate
- `cpi_yoy_pct`: Year-over-year CPI change in percentage
- `real_rate_10yr`, `real_rate_3mo`: Real interest rates (nominal minus inflation)

##### Lagged Features
Lagged versions of key indicators at 1, 3, 6, and 12 month horizons.

Indicators with `lagged=True` in `SERIES_CONFIG` receive lagged versions:
- `{col}_lag_1mo`, `{col}_lag_3mo`, `{col}_lag_6mo`, `{col}_lag_12mo`

Additional derived features are also lagged:
- `yield_curve`, `yield_curve_inverted`, `sahm_value`, `gdp_qoq_growth`
- `unemployment_accel`, `real_rate_10yr`, `real_rate_3mo`

In [2]:
def engineer_complex_features(df: pd.DataFrame) -> pd.DataFrame:
    """Derives all engineered features from the raw indicator dataset.

    Includes config-driven transformations (log differences, first differences,
    amplitude deviations), GDP growth, technical recession detection, SAHM Rule,
    rolling statistics, unemployment acceleration, and real interest rates.

    Args:
        df: DataFrame with (date, country) multi-index and raw indicator columns.

    Returns:
        DataFrame with additional engineered feature columns.
    """
    df = df.copy()
    periods = [(1, "1mo"), (3, "3mo"), (6, "6mo"), (12, "12mo")]

    # =========================================================================
    # CONFIG-DRIVEN FEATURE ENGINEERING
    # Applies feature operations based on SERIES_CONFIG settings
    # =========================================================================
    for indicator, config in SERIES_CONFIG.items():
        if indicator not in df.columns:
            continue

        # LOG DIFFERENCES - for level-based indicators
        # Captures growth rates with symmetric treatment of gains/losses
        if config.has_op("log_diff"):
            for period, label in periods:
                df[f"{indicator}_log_{label}"] = df.groupby(level="country")[
                    indicator
                ].transform(lambda x: np.log(x).diff(period))

        # FIRST DIFFERENCES - for rate-based indicators (already percentages)
        # Captures change in percentage points
        if config.has_op("first_diff"):
            for period, label in periods:
                df[f"{indicator}_diff_{label}"] = df.groupby(level="country")[
                    indicator
                ].diff(period)

        # AMPLITUDE DEVIATION FEATURES - for amplitude-adjusted indices centered at 100
        # These indices are normalized so 100 = long-term trend; deviation is meaningful
        if config.has_op("amplitude_deviation"):
            # Deviation from baseline (100)
            df[f"{indicator}_dev"] = df[indicator] - 100
            # Momentum (change over time)
            for period, label in periods:
                df[f"{indicator}_diff_{label}"] = df.groupby(level="country")[
                    indicator
                ].diff(period)

    # =========================================================================
    # GDP GROWTH & TECHNICAL RECESSION
    # =========================================================================
    # Quarter over quarter GDP growth rate (GDP is quarterly data)
    df["gdp_qoq_growth"] = (
        df.groupby(level="country")["real_gdp"].pct_change(periods=3) * 100
    )

    # Technical recession: two consecutive quarters of negative GDP growth
    df["technical_rec"] = (df["gdp_qoq_growth"] < 0) & (
        df.groupby(level="country")["gdp_qoq_growth"].shift(3) < 0
    )

    # =========================================================================
    # SAHM RULE
    # Signals recession when 3-month avg unemployment rises 0.5pp+ above
    # its 12-month low. Useful as a real-time recession indicator.
    # =========================================================================
    df["unemployment_3mo_avg"] = (
        df.groupby(level="country")["unemployment_rate"]
        .rolling(window=3, min_periods=3)
        .mean()
        .droplevel(0)
    )
    df["unemployment_12mo_min"] = (
        df.groupby(level="country")["unemployment_3mo_avg"]
        .rolling(window=12, min_periods=12)
        .min()
        .droplevel(0)
    )
    df["sahm_value"] = df["unemployment_3mo_avg"] - df["unemployment_12mo_min"]

    # =========================================================================
    # ROLLING STATISTICS - for volatility indicators
    # Captures sustained stress vs temporary spikes
    # =========================================================================
    for indicator, config in SERIES_CONFIG.items():
        if indicator not in df.columns:
            continue
        if config.has_op("rolling_stats"):
            for window in [3, 6, 12]:
                df[f"{indicator}_rolling_mean_{window}mo"] = (
                    df.groupby(level="country")[indicator]
                    .rolling(window=window, min_periods=window)
                    .mean()
                    .droplevel(0)
                )
                df[f"{indicator}_rolling_std_{window}mo"] = (
                    df.groupby(level="country")[indicator]
                    .rolling(window=window, min_periods=window)
                    .std()
                    .droplevel(0)
                )

    # =========================================================================
    # UNEMPLOYMENT ACCELERATION
    # Second derivative captures labor market inflection points
    # =========================================================================
    df["unemployment_accel"] = (
        df.groupby(level="country")["unemployment_rate"].diff().diff()
    )

    # =========================================================================
    # REAL INTEREST RATES
    # Nominal rate minus inflation
    # Using arithmetic YoY CPI change for consistency with how yields are quoted
    # =========================================================================
    df["cpi_yoy_pct"] = df.groupby(level="country")["cpi"].pct_change(periods=12) * 100
    df["real_rate_10yr"] = df["10_yr_yld"] - df["cpi_yoy_pct"]
    df["real_rate_3mo"] = df["3_mo_yld"] - df["cpi_yoy_pct"]

    return df

In [3]:
def add_lagged_features(df: pd.DataFrame) -> pd.DataFrame:
    """Adds lagged versions of key features for recession prediction.

    Lagged features capture past levels of indicators, enabling the model to
    learn leading patterns where indicators precede recessions by months.

    Args:
        df: DataFrame with (date, country) multi-index and engineered features.

    Returns:
        DataFrame with additional lagged feature columns.
    """
    df = df.copy()

    lag_periods = [(1, "1mo"), (3, "3mo"), (6, "6mo"), (12, "12mo")]

    # =========================================================================
    # COLLECT COLUMNS TO LAG FROM SERIES_CONFIG
    # =========================================================================
    cols_to_lag = []

    for indicator, config in SERIES_CONFIG.items():
        if config.lagged and indicator in df.columns:
            cols_to_lag.append(indicator)
            # Also add deviation columns for amplitude_deviation features
            if config.has_op("amplitude_deviation"):
                dev_col = f"{indicator}_dev"
                if dev_col in df.columns:
                    cols_to_lag.append(dev_col)
            # Also add rolling mean columns for rolling_stats features
            if config.has_op("rolling_stats"):
                for window in [3, 6, 12]:
                    rolling_col = f"{indicator}_rolling_mean_{window}mo"
                    if rolling_col in df.columns:
                        cols_to_lag.append(rolling_col)

    # =========================================================================
    # DERIVED FEATURES TO LAG (not in SERIES_CONFIG)
    # These are computed features that should also be lagged
    # =========================================================================
    derived_lag_cols = [
        "yield_curve",           # Inversion precedes recession by 6-18 months
        "yield_curve_inverted",  # Binary signal of inversion state
        "sahm_value",            # Designed as early recession signal
        "gdp_qoq_growth",        # GDP contraction leads recession dating
        "unemployment_accel",    # Acceleration signals inflection points
        "real_rate_10yr",        # Real 10-year rate
        "real_rate_3mo",         # Real 3-month rate
    ]
    cols_to_lag.extend([c for c in derived_lag_cols if c in df.columns])

    # =========================================================================
    # APPLY LAGS
    # =========================================================================
    for col in cols_to_lag:
        for period, label in lag_periods:
            df[f"{col}_lag_{label}"] = df.groupby(level="country")[col].shift(period)

    return df

In [4]:
# Apply complex feature engineering and lagged features
df = engineer_complex_features(df)
df = add_lagged_features(df)

print(f"Final dataset shape: {df.shape}")
print(f"Total features: {len(df.columns)}")
print(f"\nNew features added:")
print(f"  Complex features: gdp_qoq_growth, technical_rec, sahm_value, unemployment_accel, cpi_yoy_pct, real_rate_10yr, real_rate_3mo")
print(f"  Rolling statistics: {len([c for c in df.columns if 'rolling' in c])} columns")
print(f"  Lagged features: {len([c for c in df.columns if '_lag_' in c])} columns")

Final dataset shape: (5508, 237)
Total features: 237

New features added:
  Complex features: gdp_qoq_growth, technical_rec, sahm_value, unemployment_accel, cpi_yoy_pct, real_rate_10yr, real_rate_3mo
  Rolling statistics: 54 columns
  Lagged features: 116 columns


In [5]:
# Export preprocessed data
df.to_csv("data/milestone_ii_preprocessed.csv")
df.to_pickle("data/milestone_ii_preprocessed.pkl")

print("Preprocessed data exported successfully:")
print("  - CSV: data/milestone_ii_preprocessed.csv")
print("  - Pickle: data/milestone_ii_preprocessed.pkl")

Preprocessed data exported successfully:
  - CSV: data/milestone_ii_preprocessed.csv
  - Pickle: data/milestone_ii_preprocessed.pkl
